# Night-time lights — reproducible pipeline

**This notebook uses two kernels, bridged by Google Drive.**

1. **Part 1 (Python)** mounts Google Drive, then exports monthly VIIRS
   rasters from Google Earth Engine into a Drive folder with
   `Export.image.toDrive`.
2. You then **change the Colab runtime to Julia** at the banner. Colab wipes
   `/content` on a runtime switch — but **Google Drive is mounted storage, so
   the exported rasters survive**.
3. **Part 2 (Julia)** reads the rasters back from the mounted Drive, cleans
   them with `NighttimeLights.clean_complete` (the PSTT2021 pipeline), and
   aggregates to districts.

**Output:** `viirs_monthly.csv` (written to the Drive folder) — one row per
(district, month).


---
# Part 1 · Python — export VIIRS rasters to Google Drive

Run these cells with the default **Python 3** runtime.


## 1 · Mount Drive + Earth Engine

In [ ]:
!pip install -q -U earthengine-api

from google.colab import drive
drive.mount("/content/drive")          # <- the bridge across the kernel switch

import ee
ee.Authenticate()

PROJECT = "gee-ntl-470405"             # <-- your GEE Cloud project id
ee.Initialize(project=PROJECT)
print("Drive mounted, Earth Engine ready.")

## 2 · Parameters

`EXPORT_SCALE_M` is the **resampling knob**. VIIRS is 500 m native; 1000 m
halves each axis (~1/4 the pixels, ~1/4 the RAM the Julia step needs). Use
500 for native resolution, 2000 if memory is tight — the whole cleaned India
cube must fit in RAM in Part 2.

`DRIVE_FOLDER` is created under *My Drive*; it is where the rasters land and,
later, where Part 2 reads them from and writes the output CSV.


In [ ]:
START = (2014, 1)            # first (year, month), inclusive
END   = (2025, 12)           # last  (year, month), inclusive

EXPORT_SCALE_M = 1000        # resampling knob: 500 = native, 1000 = 1/2, 2000 = 1/4

DRIVE_FOLDER = "India-Built-and-Lit-VIIRS"
BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
VIIRS_COLL   = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"

## 3 · Export region (bounding box of all districts, from GitHub)

In [ ]:
import json, urllib.request

with urllib.request.urlopen(BOUNDARY_URL) as resp:
    gj = json.load(resp)
feats = [ee.Feature(ee.Geometry(ft["geometry"], proj="EPSG:4326", geodesic=False))
         for ft in gj["features"]]
region = ee.FeatureCollection(feats).geometry().bounds()
print("export region ready")

## 4 · Queue one 2-band GeoTIFF per month → Drive

Each export is a 2-band GeoTIFF: band 1 = `avg_rad` (radiance), band 2 =
`cf_cvg` (cloud-free observation count) — `clean_complete` needs both.


In [ ]:
def months(start, end):
    y, m = start
    while (y, m) <= end:
        yield y, m
        m += 1
        if m == 13:
            m, y = 1, y + 1


tasks = {}
for y, m in months(START, END):
    start = ee.Date.fromYMD(y, m, 1)
    img = (ee.ImageCollection(VIIRS_COLL)
           .filterDate(start, start.advance(1, "month"))
           .select(["avg_rad", "cf_cvg"])
           .first())
    if img is None:
        continue
    desc = f"viirs_{y}_{m:02d}"
    t = ee.batch.Export.image.toDrive(
        image=ee.Image(img).toFloat(), description=desc,
        folder=DRIVE_FOLDER, fileNamePrefix=desc, region=region,
        scale=EXPORT_SCALE_M, crs="EPSG:4326", maxPixels=1e13)
    t.start()
    tasks[desc] = t
print(f"queued {len(tasks)} monthly exports to Drive folder '{DRIVE_FOLDER}'")

## 5 · Wait for the export tasks

In [ ]:
import time

while True:
    states = [t.status()["state"] for t in tasks.values()]
    n_done = sum(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states)
    print(f"{n_done}/{len(states)} done")
    if n_done == len(states):
        break
    time.sleep(60)
print("All exports finished — rasters are in My Drive /", DRIVE_FOLDER)

---
# ⚠️  SWITCH THE COLAB RUNTIME TO JULIA NOW

**Runtime → Change runtime type → Julia**, then **Connect**.

Colab clears `/content` on the switch, but the VIIRS rasters are on **Google
Drive**, which you re-mount in the first Julia cell below. Nothing is lost.

---
# Part 2 · Julia — clean + aggregate


## 6 · Install Julia packages

In [ ]:
using Pkg
Pkg.add(["Rasters", "ArchGDAL", "NighttimeLights", "DataFrames", "CSV",
         "Dates", "Statistics", "Plots"])
using Rasters, ArchGDAL, NighttimeLights
using DataFrames, CSV, Dates, Statistics, Plots
println("Julia packages ready")

## 7 · Re-mount Google Drive

The runtime switch gave us a fresh VM, so mount Drive again. The VIIRS
rasters exported in Part 1 are waiting in the Drive folder.


In [ ]:
# Drive mounts through Python; call it from Julia via PyCall-free shell-out.
run(`python3 -c "from google.colab import drive; drive.mount('/content/drive')"`)

const DRIVE_FOLDER = "India-Built-and-Lit-VIIRS"
const RASTER_DIR   = joinpath("/content/drive/MyDrive", DRIVE_FOLDER)
const OUT          = joinpath(RASTER_DIR, "viirs_monthly.csv")
const BG_THRESHOLD = 4
const BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
@assert isdir(RASTER_DIR) "Drive folder not found - is Drive mounted?"
println("raster dir: ", RASTER_DIR)

## 8 · Load the monthly TIFs into datacubes

Each GeoTIFF has 2 bands — band 1 `avg_rad`, band 2 `cf_cvg`. They are at the
resolution chosen by `EXPORT_SCALE_M` in Part 1; each slice is materialised
with `read()` so the combined cube is a plain in-memory array.


In [ ]:
files = sort(filter(f -> endswith(f, ".tif"), readdir(RASTER_DIR)))
isempty(files) && error("no TIFs in $RASTER_DIR - did Part 1 finish?")

function tdate(f)
    m = match(r"viirs_(\d{4})_(\d{2})\.tif", f)
    Date(parse(Int, m[1]), parse(Int, m[2]))
end
dates = tdate.(files)

rad = [read(Raster(joinpath(RASTER_DIR, f); lazy=true))[Band=1] for f in files]
cf  = [read(Raster(joinpath(RASTER_DIR, f); lazy=true))[Band=2] for f in files]

rad_cube = Rasters.combine(RasterSeries(rad, Ti(dates)), Ti)
cf_cube  = Rasters.combine(RasterSeries(cf,  Ti(dates)), Ti)
println("rad cube: ", size(rad_cube), "  (", length(files), " months)")

## 9 · Clean with `NighttimeLights.clean_complete`

Runs the full PSTT2021 pipeline (NA recoding, background-noise masking,
stable-pixel detection, Hampel outlier removal, cloud-bias correction,
interpolation) over the whole India cube. `bgthreshold=4` suppresses
low-level rural noise that otherwise dominates district sums.


In [ ]:
cleaned = clean_complete(rad_cube, cf_cube; bgthreshold=BG_THRESHOLD)
println("cleaned cube: ", size(cleaned))

## 10 · Aggregate cleaned pixels to districts

In [ ]:
gjson_path = download(BOUNDARY_URL)      # SHRUG districts from GitHub

districts = NamedTuple[]
ArchGDAL.read(gjson_path) do ds
    layer = ArchGDAL.getlayer(ds, 0)
    for feat in layer
        geom = ArchGDAL.getgeom(feat)
        geom === nothing && continue
        push!(districts, (
            pc11_s_id = string(ArchGDAL.getfield(feat, "pc11_s_id")),
            pc11_d_id = string(ArchGDAL.getfield(feat, "pc11_d_id")),
            d_name    = string(ArchGDAL.getfield(feat, "d_name")),
            geom      = geom))
    end
end
println(length(districts), " districts")

ts = collect(dims(cleaned, Ti))
rows = NamedTuple[]
for (i, d) in enumerate(districts)
    i % 50 == 0 && println("  ", i, "/", length(districts))
    local masked
    try
        masked = Rasters.mask(Rasters.crop(cleaned; to=d.geom); with=d.geom)
    catch e
        @warn "skip $(d.pc11_d_id) ($(d.d_name)): $(sprint(showerror, e))"
        continue
    end
    for t in ts
        sl = view(masked, Ti=At(t))
        n  = count(!ismissing, sl)
        s  = n == 0 ? missing : sum(skipmissing(sl))
        push!(rows, (pc11_s_id=d.pc11_s_id, pc11_d_id=d.pc11_d_id,
                     d_name=d.d_name, year=year(t), month=month(t),
                     date=Date(t),
                     sum_radiance = s,
                     mean_radiance = n == 0 ? missing : s / n,
                     n_pixels = n))
    end
end
panel = DataFrame(rows)
sort!(panel, [:pc11_s_id, :pc11_d_id, :year, :month])
nrow(panel)

## 11 · Write `viirs_monthly.csv` to Drive

In [ ]:
CSV.write(OUT, panel)
println("wrote ", nrow(panel), " rows -> ", OUT)
println("(it's in My Drive / ", DRIVE_FOLDER, " - download it from there)")
first(panel, 8)

## 12 · Plot — national monthly night-time lights

In [ ]:
agg = combine(groupby(dropmissing(panel, :sum_radiance), :date),
               :sum_radiance => sum => :total)
sort!(agg, :date)

plot(agg.date, agg.total;
     lw=2, color=:orange, legend=false,
     xlabel="Month", ylabel="Sum radiance (nW cm^-2 sr^-1)",
     title="India - total VIIRS night-time lights by month",
     size=(900, 400))